# Current Trends in Long-horizon manipulation task planning and WMs

## Query openalex

In [1]:
from pyalex import Works
import pandas as pd
from tqdm import tqdm

# Configuration
SEARCH_TERMS = [
    "robotic world models",
    "long horizon robot manipulation planning",
    "robot manipulation planning",
    "JEPA world models robotic manipulation",
    "latent dynamics robot manipulation planning",
]

START_YEAR = 2021
RESULTS_PER_QUERY = 50

def reconstruct_abstract(inverted_index):
    if not inverted_index:
        return None
    words = []
    for word, positions in inverted_index.items():
        for pos in positions:
            words.append((pos, word))
    words.sort()
    return " ".join(
        [word for _, word in words]
    )

# Collect papers
papers = {}
for term in SEARCH_TERMS:
    print(f"\nSearching: {term}")
    results = (
        Works()
        .search(term)
        .filter(
            publication_year=f">{START_YEAR-1}"
        )
        .get(
            per_page=RESULTS_PER_QUERY
        )
    )

    for paper in tqdm(results):
        pid = paper["id"]
        if pid in papers:
            continue
        # Venue
        venue = None
        if paper.get("primary_location"):
            source = paper["primary_location"].get("source")

            if source:
                venue = source.get("display_name")
        # Abstract
        abstract = reconstruct_abstract(
            paper.get(
                "abstract_inverted_index"
            )
        )
        # Open access
        oa = paper.get(
            "open_access",
            {}
        )
        papers[pid] = {

            "openalex_id": pid,

            "title": paper.get(
                "display_name"
            ),

            "year": paper.get(
                "publication_year"
            ),

            "venue": venue,

            "doi": paper.get(
                "doi"
            ),

            "citations": paper.get(
                "cited_by_count"
            ),

            "abstract": abstract,

            "is_open_access": oa.get(
                "is_oa"
            ),

            "openalex_oa_url": oa.get(
                "oa_url"
            ),

            "type": paper.get(
                "type"
            )
        }
# Save
df = pd.DataFrame(
    papers.values()
)
df = df.sort_values(
    [
        "year",
        "citations"
    ],
    ascending=[
        False,
        False
    ]
)

df.to_csv(
    "papers_openalex.csv",
    index=False
)
print(
    f"\nCollected {len(df)} papers"
)
print(
    df[
        [
            "title",
            "year",
            "citations"
        ]
    ].head(10)
)


Searching: robotic world models


100%|███████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 21007.23it/s]



Searching: long horizon robot manipulation planning


100%|███████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 37429.09it/s]



Searching: robot manipulation planning


...


Searching: JEPA world models robotic manipulation


100%|███████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 21368.98it/s]



Searching: latent dynamics robot manipulation planning


100%|███████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 45819.36it/s]


Collected 214 papers
                                                 title  year  citations
140  A Survey on Vision--Language--Action Models fo...  2026         11
155  Large Language Models and Cognitive Science: A...  2026          2
178  Towards efficient real-time video motion trans...  2026          1
147        World Value Models for Robotic Manipulation  2026          0
148  Demo-JEPA: Joint-Embedding Predictive Architec...  2026          0
149  Demo-JEPA: Joint-Embedding Predictive Architec...  2026          0
154  DynaWM: A Base-VLA-Guided World Foundation Mod...  2026          0
156  VLA-JEPA: Enhancing Vision-Language-Action Mod...  2026          0
158  DynaWM: A Base-VLA-Guided World Foundation Mod...  2026          0
160  Subspace-Decomposed JEPAs: Disentangling Progr...  2026          0


## Get PDF source for download (check if it exists)

In [2]:
import requests
import re
INPUT_FILE = "papers_openalex.csv"
OUTPUT_FILE = "papers_with_pdf.csv"

# Extract arXiv ID
def find_arxiv_id(text):

    if not text:
        return None

    match = re.search(
        r"(?:arxiv\.org/abs/|arXiv:)(\d+\.\d+)",
        text,
        re.IGNORECASE
    )

    if match:
        return match.group(1)

    return None

# Query Unpaywall
def get_unpaywall_pdf(doi):

    if not doi:
        return None

    url = (
        f"https://api.unpaywall.org/v2/"
        f"{doi}"
        f"?email=your_email@example.com"
    )

    try:

        response = requests.get(
            url,
            timeout=10
        )

        if response.status_code != 200:
            return None


        data = response.json()


        best = data.get(
            "best_oa_location"
        )


        if best:

            return (
                best
                .get("url_for_pdf")
            )


    except Exception:

        return None

# arXiv PDF lookup
def get_arxiv_pdf(arxiv_id):

    if not arxiv_id:
        return None

    return (
        f"https://arxiv.org/pdf/{arxiv_id}.pdf"
    )

# Main
df = pd.read_csv(INPUT_FILE)


pdf_urls = []
sources = []

for _, row in tqdm(
    df.iterrows(),
    total=len(df)
):
    pdf = None
    source = None
    # OpenAlex OA URL

    if (
        pd.notna(
            row.get(
                "openalex_oa_url"
            )
        )
    ):

        pdf = row["openalex_oa_url"]
        source = "openalex"

    # Unpaywall
    if pdf is None:

        pdf = get_unpaywall_pdf(
            row.get("doi")
        )

        if pdf:
            source = "unpaywall"

    # arXiv
    if pdf is None:

        arxiv_id = find_arxiv_id(
            str(row["title"])
            + " "
            + str(row.get("doi"))
        )
        pdf = get_arxiv_pdf(
            arxiv_id
        )
        if pdf:
            source = "arxiv"
    pdf_urls.append(pdf)
    sources.append(source)
df["pdf_url"] = pdf_urls
df["pdf_source"] = sources
df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(
    "Finished PDF source discovery"
)
print(
    df["pdf_source"].value_counts()
)

100%|████████████████████████████████████████████████████████████████████████████████| 214/214 [00:29<00:00,  7.37it/s]

Finished PDF source discovery
pdf_source
openalex    172
Name: count, dtype: int64


## Filter Papers by relevance

In [3]:
INPUT_FILE = "papers_with_pdf.csv"
OUTPUT = "papers_filtered.csv"
df = pd.read_csv(INPUT_FILE)

# Terms that indicate robotics relevance
robot_terms = [
    "robot",
    "robotic",
    "manipulation",
    "embodied",
    "agent",
    "arm",
    "grasp",
    "dexterous",
    "sawyer",
    "franka",
    "simulator",
    "metaworld",
    "rlbench",
    "calvin"
]


# Terms that indicate planning/world model relevance
planning_terms = [
    "planning",
    "planner",
    "trajectory",
    "world model",
    "latent dynamics",
    "predictive model",
    "model-based",
    "decision making",
    "control",
    "policy"
]


def contains_terms(text, terms):

    if pd.isna(text):
        return False

    text = text.lower()

    return any(
        t in text
        for t in terms
    )

def relevance(row):

    text = (
        str(row["title"])
        + " "
        + str(row["abstract"])
    )

    return (
        contains_terms(text, robot_terms)
        and
        contains_terms(text, planning_terms)
    )
df["relevant"] = df.apply(
    relevance,
    axis=1
)
filtered = df[
    df["relevant"]
]
filtered.to_csv(
    OUTPUT,
    index=False
)
print(
    f"{len(filtered)} / {len(df)} papers retained"
)

df_fil = pd.read_csv("papers_filtered.csv")

print(df_fil[["title", "pdf_url"]].head(10))

155 / 214 papers retained
                                               title  \
0  A Survey on Vision--Language--Action Models fo...   
1        World Value Models for Robotic Manipulation   
2  Demo-JEPA: Joint-Embedding Predictive Architec...   
3  Demo-JEPA: Joint-Embedding Predictive Architec...   
4  DynaWM: A Base-VLA-Guided World Foundation Mod...   
5  VLA-JEPA: Enhancing Vision-Language-Action Mod...   
6  DynaWM: A Base-VLA-Guided World Foundation Mod...   
7  Subspace-Decomposed JEPAs: Disentangling Progr...   
8  WEAVER, Better, Faster, Longer: An Effective W...   
9  Mem-World: Memory-Augmented Action-Conditioned...   

                                     pdf_url  
0           https://arxiv.org/pdf/2405.14093  
1           https://arxiv.org/pdf/2606.24742  
2           https://arxiv.org/pdf/2605.20811  
3  https://doi.org/10.48550/arxiv.2605.20811  
4           https://arxiv.org/pdf/2607.02604  
5           https://arxiv.org/pdf/2602.10098  
6  https://doi.org/10.48550/

In [4]:
df = pd.read_csv("papers_filtered.csv")

for i, title in enumerate(df["title"]):
    print(i, title)

0 A Survey on Vision--Language--Action Models for Embodied AI
1 World Value Models for Robotic Manipulation
2 Demo-JEPA: Joint-Embedding Predictive Architecture for One-shot Cross-Embodiment Imitation
3 Demo-JEPA: Joint-Embedding Predictive Architecture for One-shot Cross-Embodiment Imitation
4 DynaWM: A Base-VLA-Guided World Foundation Model for Moving-Object Manipulation
5 VLA-JEPA: Enhancing Vision-Language-Action Model with Latent World Model
6 DynaWM: A Base-VLA-Guided World Foundation Model for Moving-Object Manipulation
7 Subspace-Decomposed JEPAs: Disentangling Progression and Content in Latent World Models
8 WEAVER, Better, Faster, Longer: An Effective World Model for Robotic Manipulation
9 Mem-World: Memory-Augmented Action-Conditioned World Models for Persistent Robot Manipulation
10 Causal-JEPA: Learning World Models through Object-Level Latent Interventions
11 JEPA-VLA: Video Predictive Embedding is Needed for VLA Models
12 Drive-JEPA: Video JEPA Meets Multimodal Trajector

In [5]:
# Create a review column

df = pd.read_csv("papers_filtered.csv")

df["include"] = ""

df.to_csv(
    "papers_review.csv",
    index=False
)

## Manual Review

_from here I went through the current list of publications and marked those which seem irrelevant (e.g. navigation tasks, soft robotics etc) with 'n' for further filtering_

In [7]:
# Filter post-review
df = pd.read_csv("papers_review.csv")

df_clean = df[
    df["include"] != "n"
]

df_clean.to_csv(
    "papers_final.csv",
    index=False
)

print(
    f"Kept {len(df_clean)} papers"
)

Kept 136 papers


# Download the pdfs

In [9]:
import os

INPUT_FILE = "papers_final.csv"

PDF_FOLDER = "data/pdfs"


os.makedirs(
    PDF_FOLDER,
    exist_ok=True
)

# Filename helper
def clean_filename(name):

    invalid = [
        "/",
        "\\",
        ":",
        "*",
        "?",
        "\"",
        "<",
        ">",
        "|"
    ]

    for char in invalid:
        name = name.replace(
            char,
            "_"
        )
    return name[:120]

# Download
df = pd.read_csv(
    INPUT_FILE
)
success = 0
for _, row in tqdm(
    df.iterrows(),
    total=len(df)
):
    url = row["pdf_url"]
    if pd.isna(url):
        continue
    filename = (
        clean_filename(
            row["title"]
        )
        +
        ".pdf"
    )
    filepath = os.path.join(
        PDF_FOLDER,
        filename
    )
    # Skip existing
    if os.path.exists(filepath):
        success += 1
        continue
    try:
        r = requests.get(
            url,
            timeout=30,
            headers={
                "User-Agent":
                "Mozilla/5.0"
            }
        )
        r.raise_for_status()
        # Check actually got PDF
        if (
            "application/pdf"
            not in r.headers.get(
                "Content-Type",
                ""
            )
        ):
            print(
                f"Not PDF: {row['title']}"
            )

            continue
        with open(
            filepath,
            "wb"
        ) as f:

            f.write(
                r.content
            )
        success += 1

    except Exception as e:
        print(
            f"\nFailed: {row['title']}"
        )

        print(e)

print(
    f"\nDownloaded {success} PDFs"
)

  2%|█▊                                                                                | 3/136 [00:01<01:12,  1.85it/s]

Not PDF: Demo-JEPA: Joint-Embedding Predictive Architecture for One-shot Cross-Embodiment Imitation


 12%|██████████▏                                                                      | 17/136 [00:17<01:30,  1.32it/s]


Failed: Embodied AI: From LLMs to World Models
403 Client Error: Forbidden for url: https://www.techrxiv.org/doi/pdf/10.36227/techrxiv.175977432.27129012


 13%|██████████▋                                                                      | 18/136 [00:18<01:22,  1.42it/s]

...


Failed: SyDeBO: Symbolic-Decision-Embedded Bilevel Optimization for Long-Horizon Manipulation in Dynamic Environments
418 Client Error: Unknown Code for url: https://ieeexplore.ieee.org/ielx7/6287639/6514899/09537786.pdf


 99%|██████████████████████████████████████████████████████████████████████████████▊ | 134/136 [04:54<00:02,  1.46s/it]


Failed: Dynamics Learning With Object-Centric Interaction Networks for Robot Manipulation
418 Client Error: Unknown Code for url: https://ieeexplore.ieee.org/ielx7/6287639/9312710/09420758.pdf


100%|████████████████████████████████████████████████████████████████████████████████| 136/136 [05:07<00:00,  2.26s/it]


Failed: Extended Task and Motion Planning of Long-horizon Robot Manipulation.
('Connection broken: IncompleteRead(2097152 bytes read, 5657216 more expected)', IncompleteRead(2097152 bytes read, 5657216 more expected))

Downloaded 56 PDFs


# Validate the PDFs

In [10]:

from pathlib import Path
import fitz  # pymupdf


PDF_DIR = "data/pdfs"


valid = []
failed = []


for pdf_file in Path(PDF_DIR).glob("*.pdf"):

    try:
        doc = fitz.open(pdf_file)

        pages = len(doc)

        if pages < 2:
            failed.append(
                (pdf_file.name, "too few pages")
            )
        else:
            valid.append(pdf_file.name)

        doc.close()

    except Exception as e:

        failed.append(
            (pdf_file.name, str(e))
        )


print(f"Valid PDFs: {len(valid)}")
print(f"Failed PDFs: {len(failed)}")


if failed:
    print("\nFailures:")
    for f in failed:
        print(f)

Valid PDFs: 63
Failed PDFs: 0


# Create Text Files

In [11]:
PDF_DIR = Path("data/pdfs")
TEXT_DIR = Path("data/text")

TEXT_DIR.mkdir(
    exist_ok=True
)


for pdf_file in PDF_DIR.glob("*.pdf"):

    print(
        f"Processing {pdf_file.name}"
    )

    doc = fitz.open(pdf_file)

    text = ""

    for page in doc:
        text += page.get_text()

    doc.close()


    output = (
        TEXT_DIR /
        pdf_file.with_suffix(".txt").name
    )


    with open(
        output,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(text)

Processing JEPA-VLA_ Video Predictive Embedding is Needed for VLA Models.pdf
Processing IKEA Furniture Assembly Environment for Long-Horizon Complex Manipulation Tasks.pdf
Processing A Step Towrad World Models: A Survey on Robotic Manipulation.pdf
Processing FurnitureBench_ Reproducible Real-World Benchmark for Long-Horizon Complex Manipulation.pdf
Processing Foundation models in robotics_Applications_challenges_and the future.pdf
Processing Mesh-based Dynamics with Occlusion Reasoning for Cloth Manipulation.pdf
Processing WEAVER, Better, Faster, Longer_ An Effective World Model for Robotic Manipulation.pdf
Processing DINO-WM_ World Models on Pre-trained Visual Features enable Zero-shot Planning.pdf
Processing Dexterous Manipulation for Multi-Fingered Robotic Hands With Reinforcement Learning_ A Review.pdf
Processing VoxPoser_ Composable 3D Value Maps for Robotic Manipulation with Language Models.pdf
Processing Subspace-Decomposed JEPAs_ Disentangling Progression and Content in Latent 

## Utilize LLM for dataset generation

In [ ]:
import pandas as pd